# Micrometeorology: WRF Extração & Comparação

Este notebook demonstra como carregar arquivos gigantescos do NetCDF (`wrfout_*`) gerados pelo WRF, focar em uma Coordenada Específica (Lat/Lon de uma estação na vida real) e extrair a Série Temporal interpolada para comparar.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from micrometeorology.wrf.series import extract_point_series
from solrad_correction.utils.plots import set_plot_style

set_plot_style()

## 1. Setup: Arquivos e Coordenadas Alvo
Abaixo nós mapeamos a pasta com os dados WRF e definimos a posição exata da estação que queremos simular para comparação.

In [ ]:
WRF_DATA_DIR = Path("../../data/raw/wrf_output")
wrf_files = sorted(WRF_DATA_DIR.glob("wrfout_d03_*"))  # Modifique para o domínio alvo d03, d04, etc

# Exemplo: Estação da UFBA / Salvador
ESTACAO_LAT = -12.9714
ESTACAO_LON = -38.5104

print(f"Encontrados {len(wrf_files)} arquivos WRF.")

## 2. Extração Vetorizada do Ponto Específico
A função `extract_point_series` usa a matemática de Pitágoras via NumPy para varrer toda a matriz global do WRF, achar o pixel (row, col) mais próximo da Estação, e extrair o histórico temporal apenas dali.

In [ ]:
if wrf_files:
    # Extrai Radiação (SWDOWN), Temp (T2) e Vento (U10, V10)
    df_wrf = extract_point_series(
        files=wrf_files,
        target_lat=ESTACAO_LAT,
        target_lon=ESTACAO_LON,
        variables=["SWDOWN", "T2", "U10", "V10"],
    )
    print(f"Dados extraídos com sucesso: {df_wrf.shape}")
    display(df_wrf.head())
else:
    print("Nenhum arquivo WRF encontrado. Verifique WRF_DATA_DIR.")

## 3. Comparação WRF vs Observado (Simulação)
Aqui simularemos o carregamento de dados da Estação Física (Observação Real) com os dados gerados pelo modelo meteorológico (WRF).

In [ ]:
if wrf_files and "SWDOWN" in df_wrf.columns:
    # 1. Ajuste a Temperatura (T2 vem em Kelvin, converta para Celsius)
    df_wrf["T2_Celsius"] = df_wrf["T2"] - 273.15

    plt.figure(figsize=(14, 6))

    # Plot WRF Radiação Solar no Ponto da Estação
    plt.plot(df_wrf.index, df_wrf["SWDOWN"], label="WRF: Radiação SWDOWN", alpha=0.8)

    # DICA: Se você tivesse df_estacao, você plota as observações reais aqui por cima!
    # plt.plot(df_estacao.index, df_estacao["rad_observada"], label="OBS: Real", alpha=0.5)

    plt.title(f"Modelagem WRF no Ponto Exato Lat:{ESTACAO_LAT} Lon:{ESTACAO_LON}")
    plt.ylabel("Radiação Global (W/m²)")
    plt.xlabel("Tempo")
    plt.legend()
    plt.tight_layout()
    plt.show()